# DLAI Model Merging - Multi-seed confirmatory run

This notebook runs the pre-declared seeds 7 and 123 and combines them with pilot seed 42. Hyperparameters are frozen before these new runs: Mean; Task Arithmetic at scale 0.75; and TIES at scale 1.0 with density 0.2.

No external notebook Input is required. Select **GPU T4 x2**, enable Internet, and Run All.

In [ ]:
!nvidia-smi
!python --version

## Install project and verify CUDA

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = 'https://github.com/LeuxLello/Dlai-model-merging.git'
WORKDIR = Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(WORKDIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(WORKDIR)])
sys.path.insert(0, str(WORKDIR / 'src'))
os.chdir(WORKDIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import gc, itertools, json, platform
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.diagnostics import cosine_similarity, l2_norm, sign_agreement, subtract_states
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import mean_merge, task_arithmetic, ties_merge
from dlai_merge.training import TrainConfig, train_specialist

assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
print('GPU:', GPU_NAME, '| capability:', CAPABILITY)
assert CAPABILITY[0] >= 7, 'Use GPU T4 x2, not P100.'
torch.ones(1, device='cuda').add_(1)

TASKS = ['sst2', 'imdb', 'mrpc', 'rte']
NEW_SEEDS = [7, 123]
BASE_MODEL = 'prajjwal1/bert-mini'
TRAIN_ROOT = Path('/kaggle/working/multiseed_specialists')

## Train and evaluate the two new seeds
Each task receives the exact same 400-step budget used for seed 42. Checkpoints are temporary: only compact metrics are retained after evaluation.

In [ ]:
new_merge_rows = []
new_diagnostic_rows = []
new_specialist_rows = []

for seed in NEW_SEEDS:
    print(f'\n######## SEED {seed} ########')
    summaries = {}
    for task in TASKS:
        print(f'\n===== TRAINING {task.upper()} / SEED {seed} =====')
        summaries[task] = train_specialist(TrainConfig(
            task=task, output_root=str(TRAIN_ROOT), seed=seed,
            max_train_samples=12_000, max_eval_samples=2_000,
            max_steps=400, eval_steps=100, train_batch_size=32, eval_batch_size=64,
            learning_rate=2e-5,
        ))

    encoders = {t: torch.load(TRAIN_ROOT / t / f'seed-{seed}' / 'encoder.pt', map_location='cpu', weights_only=True) for t in TASKS}
    heads = {t: torch.load(TRAIN_ROOT / t / f'seed-{seed}' / 'head.pt', map_location='cpu', weights_only=True) for t in TASKS}
    base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
    base_encoder = {k: v.detach().cpu().clone() for k, v in base_model.base_model.state_dict().items()}
    del base_model
    vectors = {t: subtract_states(encoders[t], base_encoder) for t in TASKS}

    for left, right in itertools.combinations(TASKS, 2):
        agreement = sign_agreement(vectors[left], vectors[right])
        new_diagnostic_rows.append({
            'seed': seed, 'task_a': left, 'task_b': right, 'pair': f'{left}+{right}',
            'cosine_similarity': cosine_similarity(vectors[left], vectors[right]),
            'sign_agreement': agreement, 'sign_conflict': 1.0 - agreement,
            'norm_a': l2_norm(vectors[left]), 'norm_b': l2_norm(vectors[right]),
        })

    evaluators = {t: TaskEvaluator(t, heads[t], max_eval_samples=2000, seed=seed, output_root=f'/kaggle/working/eval-{seed}') for t in TASKS}
    references = {}
    for task in TASKS:
        score = evaluators[task].evaluate(encoders[task])
        references[task] = score['primary_score']
        new_specialist_rows.append({'seed': seed, 'task': task, 'primary_metric': evaluators[task].primary_metric, **score})

    def evaluate_merge(pair, method, scale, density, merged):
        for task in pair:
            score = evaluators[task].evaluate(merged)
            reference = references[task]
            new_merge_rows.append({
                'seed': seed, 'task_a': pair[0], 'task_b': pair[1], 'pair': '+'.join(pair),
                'method': method, 'scale': scale, 'density': density, 'eval_task': task,
                **score, 'specialist_score': reference,
                'retained_ratio': score['primary_score'] / reference,
                'score_delta': score['primary_score'] - reference,
            })

    for pair in itertools.combinations(TASKS, 2):
        states = [encoders[pair[0]], encoders[pair[1]]]
        print('Frozen merges:', pair, 'seed', seed)
        evaluate_merge(pair, 'mean', 1.0, np.nan, mean_merge(base_encoder, states))
        evaluate_merge(pair, 'task_arithmetic', 0.75, np.nan, task_arithmetic(base_encoder, states, 0.75))
        evaluate_merge(pair, 'ties', 1.0, 0.2, ties_merge(base_encoder, states, 0.2, 1.0))

    del evaluators, encoders, heads, vectors, base_encoder
    gc.collect(); torch.cuda.empty_cache()
    for task in TASKS:
        shutil.rmtree(TRAIN_ROOT / task / f'seed-{seed}', ignore_errors=True)

print('New task-level merge rows:', len(new_merge_rows))
assert len(new_merge_rows) == 72

## Add frozen seed-42 rows from the committed pilot

In [ ]:
seed42_results = pd.read_csv(WORKDIR / 'results/merging_pilot_seed42/merge_results.csv')
frozen42 = seed42_results[
    ((seed42_results.method == 'mean')) |
    ((seed42_results.method == 'task_arithmetic') & np.isclose(seed42_results.scale, 0.75)) |
    ((seed42_results.method == 'ties') & np.isclose(seed42_results.scale, 1.0) & np.isclose(seed42_results.density, 0.2))
].copy()
frozen42.insert(0, 'seed', 42)
assert len(frozen42) == 36

seed42_diag = pd.read_csv(WORKDIR / 'results/merging_pilot_seed42/pair_diagnostics.csv')
seed42_diag.insert(0, 'seed', 42)
seed42_specialists = pd.read_csv(WORKDIR / 'results/merging_pilot_seed42/specialist_recheck.csv')
seed42_specialists.insert(0, 'seed', 42)

all_results = pd.concat([frozen42, pd.DataFrame(new_merge_rows)], ignore_index=True)
all_diagnostics = pd.concat([seed42_diag, pd.DataFrame(new_diagnostic_rows)], ignore_index=True)
all_specialists = pd.concat([seed42_specialists, pd.DataFrame(new_specialist_rows)], ignore_index=True)
print('All task-level merge rows:', len(all_results))
assert len(all_results) == 108

## Aggregate across tasks and seeds

In [ ]:
config_cols = ['seed', 'pair', 'task_a', 'task_b', 'method', 'scale', 'density']
per_seed_pair = (all_results.groupby(config_cols, dropna=False)
    .agg(mean_retained=('retained_ratio', 'mean'), worst_retained=('retained_ratio', 'min'), mean_delta=('score_delta', 'mean'))
    .reset_index())
final_summary = (per_seed_pair.groupby(['pair', 'method', 'scale', 'density'], dropna=False)
    .agg(retained_mean=('mean_retained', 'mean'), retained_std=('mean_retained', 'std'),
         worst_mean=('worst_retained', 'mean'), delta_mean=('mean_delta', 'mean'))
    .reset_index())
method_summary = (per_seed_pair.groupby(['seed', 'method'], as_index=False)
    .agg(mean_retained=('mean_retained', 'mean'), worst_retained=('worst_retained', 'min')))
final_summary.sort_values(['pair', 'retained_mean'], ascending=[True, False])

## Per-seed confirmatory correlations using fixed Mean merging

In [ ]:
mean_retention = per_seed_pair[per_seed_pair.method == 'mean'][['seed', 'pair', 'mean_retained']]
correlation_data = mean_retention.merge(all_diagnostics, on=['seed', 'pair'])
correlation_rows = []
for seed, group in correlation_data.groupby('seed'):
    correlation_rows.append({
        'seed': seed,
        'pearson_cosine': group.cosine_similarity.corr(group.mean_retained, method='pearson'),
        'spearman_cosine': group.cosine_similarity.corr(group.mean_retained, method='spearman'),
        'pearson_sign_conflict': group.sign_conflict.corr(group.mean_retained, method='pearson'),
    })
correlations = pd.DataFrame(correlation_rows)
correlations

## Final multi-seed figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FIGURES = Path('/kaggle/working/multiseed_figures'); FIGURES.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=correlation_data, x='cosine_similarity', y='mean_retained', hue='seed', style='pair', s=90, ax=axes[0])
sns.barplot(data=method_summary, x='method', y='mean_retained', hue='seed', ax=axes[1])
axes[0].axhline(1.0, color='gray', linestyle='--'); axes[0].set_title('Fixed Mean merging across seeds')
axes[1].axhline(1.0, color='gray', linestyle='--'); axes[1].set_title('Frozen methods across seeds')
axes[1].tick_params(axis='x', rotation=15)
fig.tight_layout(); fig.savefig(FIGURES / 'multiseed_confirmatory.png', dpi=180, bbox_inches='tight')
plt.show()

## Export compact evidence

In [ ]:
OUT = Path('/kaggle/working/multiseed_results'); OUT.mkdir(exist_ok=True)
all_results.to_csv(OUT / 'all_merge_results.csv', index=False)
all_diagnostics.to_csv(OUT / 'all_pair_diagnostics.csv', index=False)
all_specialists.to_csv(OUT / 'all_specialist_scores.csv', index=False)
per_seed_pair.to_csv(OUT / 'per_seed_pair.csv', index=False)
final_summary.to_csv(OUT / 'final_summary.csv', index=False)
method_summary.to_csv(OUT / 'method_summary.csv', index=False)
correlations.to_csv(OUT / 'correlations.csv', index=False)
shutil.copy2(FIGURES / 'multiseed_confirmatory.png', OUT / 'multiseed_confirmatory.png')
metadata = {
    'purpose': 'confirmatory multi-seed run with frozen configurations',
    'seeds': [42, 7, 123], 'tasks': TASKS,
    'frozen_methods': {'mean': {'scale': 1.0}, 'task_arithmetic': {'scale': 0.75}, 'ties': {'scale': 1.0, 'density': 0.2}},
    'gpu': GPU_NAME, 'python': platform.python_version(), 'torch': torch.__version__,
}
(OUT / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/multiseed_confirmatory_results', 'zip', OUT)
print(archive)
print(*sorted(str(p) for p in OUT.iterdir()), sep='\n')